# Lesson 62 · `Dataset` and `DataLoader` — teaching PyTorch to "eat" sound: feeding one second of speech in as batches of mel features

**Goal**: first turn one second of speech into batches of mel features, then let `Dataset` / `DataLoader` deliver them reliably to the model (the offline synthetic fallback runs end to end too).

> This lesson uses only our own `aurora.audio.mel`, never `torchaudio.transforms`. If a path or a shape goes wrong, check the working directory and the data split first.

🔗 Aurora connection: `aurora.audio.mel.mel_spectrogram()` — this lesson uses our own implementation throughout and never calls `torchaudio.transforms`.

### Why a mel spectrogram? — the ear is closer to a "logarithmic scale"

Imagine pressing two keys on a piano:

- **Low register**: 100 Hz and 200 Hz, 100 Hz apart. Can your ear tell them apart clearly? **Yes**.
- **High register**: 10000 Hz and 10100 Hz, also 100 Hz apart. Can you hear the difference? **Barely**.

Why? **Because the ear's frequency resolution is *relative* (a percentage), not absolute**.

```
the ear's "sense of distance" ∝ log(frequency)
```

This observation is quantified as the **mel scale** (mel = melody):

| Frequency (Hz) | Mel value | What it means |
|---|---|---|
| 0 | 0 | silence |
| 100 | ≈ 150 | a low A, roughly |
| 500 | ≈ 607 | lower midrange |
| 1000 | ≈ 1000 | midrange |
| 2000 | ≈ 1521 | upper midrange |
| 10000 | ≈ 3073 | high register, but mel growth has slowed |

**Formula**: $\text{Mel} = 2595 \times \log_{10}\left(1 + \frac{f}{700}\right)$

Notice anything? **The mel value grows more and more slowly** — that's the signature of a logarithm.

**In a mel spectrogram, one mel bin stands for a frequency interval the ear perceives as "equally spaced"**. Dense at low frequencies, sparse at high ones — exactly matching the unevenness of human hearing.

**Why do CNNs like 40 mel bins?**

- 513 linear FFT bins: the high end is mostly noise and hugely redundant, so the CNN needs more parameters to tell meaningless details apart → risk of overfitting.
- 40 mel bins: reorganized the way the ear perceives, so the contrast between signal and noise stands out and the classifier learns faster.

In this lesson, **what `aurora.audio.mel.mel_spectrogram()` does inside is**:
1. STFT → 513 linear frequency bins
2. pass through the mel filterbank (40 overlapping triangular filters, centers spaced on the mel scale)
3. compress 513 dimensions down to 40 while keeping the information the ear cares about



← **Previous lesson** · [L61 · `nn.Module` in practice](L61_pytorch_nn.ipynb)

> Last lesson covered **`nn.Module` in practice**: Linear / ReLU / Sequential, managing parameters and saving models.
> This lesson explores **`Dataset` and `DataLoader`**.

## This lesson's story: squeezing "one second of sound" into a map

The first hurdle in speech recognition isn't "understanding the meaning" — it's tidying waveforms of all different lengths into one uniform size.

**Two preprocessing steps** turn sound into a standard map:

```
raw waveform (sr=16000, any length)
   ↓ pad/truncate → exactly 16000 samples (1 second)
   ↓ STFT + mel filterbank → a (n_mels=40, T≈32 frames) "time–frequency plot"
   ↓ normalize → mean=0, variance=1
   ↓ into the CNN/MLP
```

**Why a mel spectrogram rather than the raw waveform**:

| Representation | Size | Information | Suits a CNN? |
|---|---|---|---|
| raw waveform | 16000 points | everything | just about, but very redundant |
| linear FFT spectrum | 513 bins × T | every frequency | high-frequency detail is wasted |
| mel spectrogram | 40 × T | perceptual frequencies | ✅ best |

**The Speech Commands dataset**: 35 keyword classes (yes/no/up/down…), each clip about 1 second, 16 kHz, ~2.3 GB.

This lesson's task: implement `extract_features(wav_path)` — read the WAV → pad/truncate → mel → dB → normalization → return a `(40, T)` Tensor.

In [ ]:
# Aurora matplotlib bootstrap
from pathlib import Path
import sys

_root = None
_cwd = Path.cwd().resolve()
for _candidate in (_cwd, *_cwd.parents):
    if (_candidate / '_matplotlib_bootstrap.py').exists():
        _root = _candidate
        break
if _root is None:
    _notebooks_dir = _cwd / 'notebooks'
    if _notebooks_dir.exists():
        for _found in _notebooks_dir.rglob('_matplotlib_bootstrap.py'):
            _root = _found.parent
            break
if _root is not None and str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from _matplotlib_bootstrap import apply as _aurora_mpl_apply
_aurora_mpl_apply()


In [ ]:
import torch
import numpy as np
from aurora.audio.io import read_wav, write_wav
from aurora.audio.mel import mel_spectrogram, power_to_db
import matplotlib.pyplot as plt
%matplotlib inline

# Dependencies: this lesson needs torch (install: pip install -e ".[speech]");
# torchaudio is needed only for the real Speech Commands data —
# without it (or without a network) the lesson switches to synthetic keyword data and still runs end to end.

## 1. The Speech Commands dataset (falls back to synthetic data offline)

Speech Commands contains 35 classes of English keywords (yes / no / up / down, etc.), each clip about **1 second** long, at a sample rate (SR) of **16 kHz**, as mono WAV files. The label is given by the subdirectory name; each file looks like:
```
data/SpeechCommands/speech_commands_v0.02/yes/0a7c2a8d_nohash_0.wav
```
35 classes, roughly 2000–4000 clips per class, about 105 000 training samples in total.

> **Download size heads-up**: the full archive is about **2.3 GB**. This lesson does **not** download it automatically — the code below uses the real data only if it finds torchaudio and the data is already on disk; otherwise it falls back automatically to a **fixed-seed synthetic keyword dataset** (each keyword is imitated by a frequency sweep in a different band), so the whole lesson runs with no network and no torchaudio.
>
> To use the real data: install torchaudio with `pip install -e ".[speech]"`, change `download=False` below to `True`, and run it once.

Both paths produce samples in exactly the same format: `(waveform_tensor(1, N), sr, label, ...)`, so none of the later cells needs to care where the data came from.

#### What does a 16 kHz sample rate actually mean?

**16 kHz = 16 000 samples / second**

So to record **exactly 1 second** of audio you need to capture:
$$\text{total samples} = \text{sample rate} \times \text{duration} = 16000 \, \text{samples/s} \times 1 \, \text{s} = 16000 \, \text{samples}$$

This is the "unit conversion" you did at school — picture apples at 16 dollars a pound: buy 1 pound and it costs 16 dollars. Here you're buying 1 second of sound, and at a sample rate of 16000 you get 16000 numbers.

The other way round, if you have a WAV file of unknown length and want to know how long it plays:
$$\text{duration (s)} = \frac{\text{number of samples}}{\text{sample rate}} = \frac{\text{len}(x)}{16000}$$



In [ ]:
# Data loading: prefer the real Speech Commands; if that fails, fall back to the synthetic keyword dataset
SR = 16000

def _synth_keyword(label: str, rng: np.random.Generator) -> np.ndarray:
    """Synthesize a 1-second keyword waveform: different label = frequency sweep in a different band + Gaussian envelope + light noise.

    Three key ideas:

    1. Frequency sweep (chirp): the frequency isn't fixed, it sweeps linearly from A Hz to B Hz.
       - "yes" uses a high sweep (2000→3500 Hz), "no" a low one (200→350 Hz)
       - so the synthetic "yes" and "no" get a natural difference in pitch

    2. Deriving the phase: in sin(2π×f×t), how do you write it if f isn't a constant but a function of time f(t)?
       - simple sweep: f_inst(t) = f_0 + (f_1 - f_0) × (t / T), where T is the total duration
       - to let sin "know the frequency is changing", integrate the frequency (accumulate it) to get the phase:
         phase(t) = 2π × ∫[0 to t] f_inst(τ) dτ
       - in the discrete case, ∫ is replaced by cumsum: np.cumsum(freq) is the accumulated phase "speed"
       - then divide by the sample rate SR to turn sample counts into time (sample spacing = 1/SR)
       → finally: phase = 2π × cumsum(freq) / SR

    3. Gaussian envelope: the loudness "breathing" over time.
       - a plain sine wave hangs at the edges (starts abruptly, stops abruptly), which the STFT picks up as "edge energy"
       - the Gaussian envelope exp(-(...)) fades smoothly at both ends, so the waveform "fades in and out"
       - centered at dur/2 (the middle of the utterance), width set by dur/4
       - result: the waveform is near 0 at head and tail and strong in the middle, much like the loudness of natural speech
    """
    band = {                      # (start frequency, end frequency) Hz
        'yes':  (2000.0, 3500.0),   # high band, imitating the high-frequency energy of /jɛs/
        'no':   (200.0, 350.0),     # low band, imitating the nasal /noʊ/
        'up':   (800.0, 1600.0),
        'down': (600.0, 300.0),     # downward sweep
    }[label]
    t = np.arange(SR) / SR
    freq = band[0] + (band[1] - band[0]) * t            # linear sweep: freq(t) = f_start + (f_end - f_start) * t
    phase = 2 * np.pi * np.cumsum(freq) / SR            # phase = 2π × ∫f dt, approximated discretely with cumsum
    dur = rng.uniform(0.5, 0.9)                          # each "utterance" has a different duration (0.5–0.9 s)
    env = np.exp(-((t - dur / 2) ** 2) / (2 * (dur / 4) ** 2))  # Gaussian envelope: centered at dur/2, width dur/4
    wav = env * np.sin(phase) + 0.02 * rng.standard_normal(SR)  # envelope modulation + faint noise
    return (0.5 * wav / np.abs(wav).max()).astype(np.float32)

def make_synthetic_ds(n_per_class: int = 20, labels=('yes', 'no', 'up', 'down'), seed: int = 0):
    """Return a list of samples in the same format as torchaudio SPEECHCOMMANDS: (waveform(1,N), sr, label)."""
    rng = np.random.default_rng(seed)
    return [
        (torch.from_numpy(_synth_keyword(label, rng)).unsqueeze(0), SR, label)
        for label in labels
        for _ in range(n_per_class)
    ]

try:
    import torchaudio
    root = './data'
    # download=False: use only data already on disk; to really download it (~2.3 GB), change this to True
    train_ds = torchaudio.datasets.SPEECHCOMMANDS(root, download=False, subset='training')
    DATA_SOURCE = 'SpeechCommands (real data)'
except Exception as e:
    train_ds = make_synthetic_ds()
    DATA_SOURCE = f'synthetic fallback data (reason: {type(e).__name__} — no torchaudio, or no local data)'

print(f'data source: {DATA_SOURCE}')
print(f'training set size: {len(train_ds)} clips')
waveform, sr, label, *_ = train_ds[0]
print(f'sample 0 → label={label}, sr={sr}, shape={tuple(waveform.shape)}')

## 2. Making everything one length: pad / truncate to 16 000 samples

Raw clips vary slightly in length (recordings are cut off unevenly). A CNN needs fixed-length input, so we make every clip exactly **16 000 samples** (1 second @ 16 kHz):

```
if len(x) < 16000:  x = pad_right(x, 16000 - len(x))   # pad zeros at the end
if len(x) > 16000:  x = x[:16000]                        # truncate
```

### Why does zero-padding produce "all-zero frames"?

It isn't magic, just a natural consequence of the STFT:

1. Your waveform has 14000 samples → pad 2000 zeros → now it's 16000 samples
2. The STFT slices it with n_fft=2048, hop_length=512, starting at 0 and jumping 512 samples at a time
3. As the window slides near the end — say [14400:16448] — every sample in it lies past the original audio's 14000-sample end: **it's all padding zeros**
4. The FFT processes that window: an all-zero window → an all-zero FFT
5. So the last few columns of the mel spectrogram are all zero (energy = 0)

**This barely affects classification** — the keyword is usually finished within the first 0.5–0.8 seconds; the all-zero frames at the end are "silence". They carry no information, but they don't interfere with feature extraction either.

In [ ]:
def pad_or_truncate(x: np.ndarray, target: int = 16000) -> np.ndarray:
    """Bring a 1-D numpy array to length target (zero-pad or truncate)."""
    if len(x) >= target:
        return x[:target]
    return np.concatenate([x, np.zeros(target - len(x), dtype=x.dtype)])

# demo
x_short = np.random.randn(14000).astype(np.float32)
x_long  = np.random.randn(17500).astype(np.float32)
print(pad_or_truncate(x_short).shape)  # (16000,)
print(pad_or_truncate(x_long).shape)   # (16000,)

## 3. Normalization: matching the global mean/variance (while keeping the dynamics over time)

Loudness varies hugely across speakers and microphones; feeding that straight into the network makes the gradients unstable. So we standardize each mel feature map:

```
X_norm = (X - mean(X)) / (std(X) + eps)
```

where `mean` and `std` are computed over **all mel × time** points of that sample (**per-sample global normalization**). `eps = 1e-6` prevents division by zero.

### Why "global" and not "per-frame"?

You might ask: wouldn't "per-frame" normalization (normalizing each time-frame column separately) also get rid of noise?

**No. There's a trap here**:

Picture two keyword samples:
1. "yes": the waveform goes from weak to strong (getting louder) — low energy in the first half, high in the second
2. "no": the reverse (getting quieter) — high energy in the first half, low in the second

**What per-frame normalization does**:
```
"yes" raw: [low, low, low, high, high, high] → per-frame: [flat, flat, flat, flat, flat, flat]
           no difference left at all — the dynamics over time are "flattened"

"no" raw:  [high, high, high, low, low, low] → per-frame: [flat, flat, flat, flat, flat, flat]
          flattened just the same
```

**The advantage of global normalization**:
```
"yes" raw: [1, 1, 1, 10, 10, 10]  →  (X - mean) / std  →  after normalizing, still "weak then strong"
"no" raw:  [10, 10, 10, 1, 1, 1]  →  (X - mean) / std  →  after normalizing, still "strong then weak"
```

**Key insight**: global normalization only changes "the absolute size and spread of the numbers"; it **doesn't change the relative shape**. And the dynamics over time (strong first half or strong second half) are precisely one of the key features a CNN uses to tell keywords apart.

This differs from BatchNorm: BatchNorm computes statistics across the batch dimension, whereas here each sample is normalized on its own, independent of batch size, so it works as-is at inference time too.

In [ ]:
def normalize(X: np.ndarray, eps: float = 1e-6) -> np.ndarray:
    """Per-sample mean/variance normalization of a 2-D mel feature map."""
    return (X - X.mean()) / (X.std() + eps)

# Demo 1: basic behavior
X = np.random.randn(40, 81).astype(np.float32) * 10 + 5
Xn = normalize(X)
print(f'Demo 1 — basic normalization')
print(f'  before: mean={X.mean():.2f}, std={X.std():.2f}')
print(f'  after:  mean={Xn.mean():.4f}, std={Xn.std():.4f}\n')

# Demo 2: why global normalization keeps the dynamics over time
print(f'Demo 2 — global vs per-frame normalization, and what happens to the dynamics over time:')
# build two samples: one "getting louder", one "getting quieter"
X_rising = np.array([
    [1, 2, 3, 10, 20, 30],      # one row over time, energy going from weak to strong
    [1, 2, 3, 10, 20, 30],      # (repeated, to imitate several mel bins)
]).astype(np.float32)
X_falling = np.array([
    [30, 20, 10, 3, 2, 1],      # the reverse: energy going from strong to weak
    [30, 20, 10, 3, 2, 1],
]).astype(np.float32)

# global normalization
X_rising_global = normalize(X_rising)
X_falling_global = normalize(X_falling)

# per-frame normalization
X_rising_perframe = (X_rising - X_rising.mean(axis=0, keepdims=True)) / (X_rising.std(axis=0, keepdims=True) + 1e-6)
X_falling_perframe = (X_falling - X_falling.mean(axis=0, keepdims=True)) / (X_falling.std(axis=0, keepdims=True) + 1e-6)

print(f'  raw "rising": {X_rising[0]}')
print(f'  after global normalization:    {np.round(X_rising_global[0], 3)}  ← still the "1→30" trend')
print(f'  after per-frame normalization: {np.round(X_rising_perframe[0], 3)}  ← flattened to all zeros: the rising trend is gone!\n')

print(f'  raw "falling": {X_falling[0]}')
print(f'  after global normalization:    {np.round(X_falling_global[0], 3)}  ← still the "30→1" trend')
print(f'  after per-frame normalization: {np.round(X_falling_perframe[0], 3)}  ← flattened too: the feature is lost just the same\n')

print(f'✅ Conclusion: global normalization ✓ keeps the dynamics over time; per-frame normalization ✗ flattens them')

## 4. ✏️ Implement `extract_features(wav_path, sr=16000, n_mels=40)`

**Signature**:
```python
def extract_features(wav_path: str, sr: int = 16000, n_mels: int = 40) -> torch.Tensor:
    # returns: a float32 Tensor of shape (n_mels, T)
```

**The 6-step implementation route** (matches the TODOs below one to one):

| Step | Operation | Input/output | Tool |
|---|---|---|---|
| 1 | read the WAV → numpy array | (N,) | `aurora.audio.io.read_wav(wav_path)` |
| 2 | assert sample rate == `sr` | check file_sr | `assert file_sr == sr` |
| 3 | pad/truncate → `sr` samples | (N,) → (16000,) | `pad_or_truncate()` |
| 4 | compute the mel spectrogram → `(T, n_mels)` | (16000,) → (T, 40) | `mel_spectrogram(..., n_fft=2048, hop_length=512)` |
| 4b | convert to a dB scale | (T, 40) → (T, 40) | `power_to_db(mel)` — the shape doesn't change, only the scale of the values |
| 5 | transpose → `(n_mels, T)` | (T, 40) → (40, T) | `.T` — matches the CNN's (height = frequency, width = time) |
| 6 | normalize + convert to a float32 Tensor | (40, T) float32 | `normalize()` + `torch.from_numpy(...).float()` |

**Why transpose in step 5?**
A PyTorch 2D CNN expects input of shape `(B, C, H, W)`. For an audio spectrogram:
- `C` (channel) = 1 (a single "grayscale" channel, unlike the 3 RGB channels of a color image)
- `H` (height) = the frequency direction = number of mel bins = 40
- `W` (width) = the time direction

So `(40, T)` is one 40×T image with frequency running up and time running across (add a channel dimension and it becomes `(1, 40, T)`); a convolution kernel as tall as the image sweeps across all the mel bins at once.

**Acceptance criteria**:
- output `dtype == torch.float32`
- `output.shape == (40, 32)` (T≈32 for 16000 samples with `n_fft=2048`, `hop_length=512`)
- non-zero (not an all-zero array)

### Why convert to a dB (decibel) scale?

The mel spectrogram outputs **energy** (power), and the values can span a huge range: from near 0 (silence) to 1000+ (loud noise). A distribution like that is very "unfriendly" to deep learning:

- a small signal (say speech = 0.1) and a large one (say background noise = 100) differ by a **factor of 1000**
- extreme values can make the network's gradients explode or vanish

**But that isn't how the ear perceives it**. "1 vs 0.1" and "100 vs 1000" are both a factor of 10, and they sound like roughly the same step (10 dB each). That's **the magic of the logarithm**.

**Definition of the decibel (dB)** for power:
$$\text{dB} = 10 \times \log_{10}\left(\frac{\text{power}}{\text{reference power}}\right)$$

The reference is usually taken as $10^{-12}$ watts (per square meter — roughly the threshold of human hearing).

**Comparing values**:
| Energy (linear) | dB value | How it feels |
|---|---|---|
| $10^{-12}$ | 0 dB | just audible |
| $10^{-10}$ | 20 dB | very quiet |
| $10^{-6}$ | 60 dB | normal conversation |
| $10^{-2}$ | 100 dB | very loud |
| $10^{2}$ | 140 dB | painful |

**See it?** A factor-of-100 difference in linear space (from $10^{-12}$ to $10^{-10}$) becomes just a 20 dB difference once converted. That gives the network a much more "balanced" data distribution, and the gradients flow more stably.

**In practice**:
```python
mel_raw = mel_spectrogram(...)  # values like [0.001, 0.01, 1, 100, ...]
mel_db = power_to_db(mel_raw)   # values like [-30, -20, 0, 20, ...], much more "moderate"
```

After the dB conversion most values fall between -60 and 40, so the gradients don't blow up because of extreme values.



### Deriving the number of time frames from the STFT parameters (one step deeper)

**The question**: you have 16000 samples and run an STFT with `n_fft=2048` and `hop_length=512` — why do you get T≈32 frames?

**The intuition**: cut the waveform into overlapping windows and run an FFT on each window.

```
time axis ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16000 samples
         ↑ window 0: [0:2048]  ← n_fft=2048 samples
         ↑ jump 512 samples
              ↑ window 1: [512:2560]
              ↑ jump 512 samples
                   ↑ window 2: [1024:3072]
                   ...
```

Each jump is `hop_length=512` samples — so when does the last complete window start?

**Formula**:
$$T = \left\lfloor \frac{N - n\_fft}{hop\_length} \right\rfloor + 1$$

Plugging in the numbers:
$$T = \left\lfloor \frac{16000 - 2048}{512} \right\rfloor + 1 = \left\lfloor \frac{13952}{512} \right\rfloor + 1 = \lfloor 27.25 \rfloor + 1 = 27 + 1 = 28$$

Wait — the formula gives 28, not 32?

The difference is padding at the edges. Like librosa's default, `aurora.audio.mel.mel_spectrogram()` uses a **centered** STFT: it reflect-pads `n_fft // 2 = 1024` samples onto each end, so that frame t is centered on sample `t × hop_length`. Then N becomes 16000 + 2048 = 18048, and $T = \lfloor (18048 - 2048)/512 \rfloor + 1 = \lfloor 31.25 \rfloor + 1 = 32$ — that is, $T = 1 + \lfloor N / hop\_length \rfloor$. Implementations that don't center give 28. The most practical habit:

**Just check the real output**: whenever you find yourself asking "why is T=32?", simply run the code:

```python
import numpy as np
from aurora.audio.mel import mel_spectrogram
x = np.random.randn(16000).astype(np.float32)
mel = mel_spectrogram(x, sample_rate=16000, n_mels=40, n_fft=2048, hop_length=512)
print(mel.shape)  # see the real (T, 40): 32 when centered, 28 when not
```

This matters more than memorizing the formula — **understand which way each parameter pushes**:
- larger `n_fft` → higher "frequency resolution" per frame (513 vs 1025 bins), but possibly fewer frames
- smaller `hop_length` → more frames (denser slicing), but more computation
- the padding strategy → affects the frame count at the ends

Remember the hierarchy: **direction of each parameter > formula > absolute value**.



In [ ]:
def extract_features(
    wav_path: str,
    sr: int = 16000,
    n_mels: int = 40,
) -> torch.Tensor:
    """Load WAV → pad/truncate → mel spectrogram → dB conversion → normalize → torch.Tensor (n_mels, T)."""
    # ✏️ TODO: 1. read the audio with read_wav()
    x, file_sr = None, None

    # ✏️ TODO: 2. assert the sample rate, handle stereo

    # ✏️ TODO: 3. pad_or_truncate to sr samples

    # ✏️ TODO: 4. mel_spectrogram(x, sample_rate=sr, n_mels=n_mels, n_fft=2048, hop_length=512) → (T, n_mels)

    # ✏️ TODO: 4b. power_to_db(mel) → dB scale (from aurora.audio.mel import power_to_db)

    # ✏️ TODO: 5. transpose → (n_mels, T)

    # ✏️ TODO: 6. normalize + convert to torch.Tensor
    raise NotImplementedError

In [ ]:
# Take one "yes" sample from the training set, write it to a temporary WAV, and run the full extract_features pipeline
import tempfile, os

waveform, file_sr, label, *_ = next(s for s in train_ds if s[2] == 'yes')
wav_np = waveform.squeeze().numpy()

# write it to disk and read it back with aurora's own write_wav (no soundfile needed); the finally block cleans up the temp file
tmp_path = None
try:
    with tempfile.NamedTemporaryFile(suffix='.wav', delete=False) as tmp:
        tmp_path = tmp.name
    write_wav(tmp_path, wav_np, file_sr)
    try:
        feat = extract_features(tmp_path, sr=16000, n_mels=40)
        # stronger assertion: check both the n_mels and the T dimension
        assert feat.shape == (40, 32), (
            f"expected (40, 32), got {tuple(feat.shape)}\n"
            "  → check that mel_spectrogram was given n_fft=2048, hop_length=512 and that you applied the .T transpose"
        )
        assert feat.dtype == torch.float32, "dtype should be float32"
        print(f'✅ feat.shape = {tuple(feat.shape)}  dtype = {feat.dtype}')
    except (NotImplementedError, TypeError):
        print("⚠️ extract_features isn't implemented yet — finish TODOs 1–6, then run this cell again")
finally:
    if tmp_path and os.path.exists(tmp_path):
        os.unlink(tmp_path)

## 5. Parameter experiment: comparing the average mel features of "yes" and "no"

**Parameters**: `n_mels=40`, `n_samples=10` (10 clips per class)

**What you should see**:
- **Real Speech Commands**: the vowel /ɛ/ in "yes" makes the mid-to-high bands (upper half of the plot) stronger; "no" ends on a low-frequency nasal, so the low bands (lower half of the plot) carry more sustained energy.
- **Synthetic fallback data**: the synthetic "yes" is a 2000→3500 Hz sweep, with its energy concentrated in the upper half of the mel plot; the synthetic "no" is a 200→350 Hz low tone, with its energy concentrated in the bottom few mel bins.
- Whichever data source you use, the energy blocks sit in clearly different places in the two average plots — and that's exactly what lets a CNN tell them apart.

In [ ]:
import tempfile, os

def collect_features(dataset, keyword: str, n: int = 10):
    feats = []
    for waveform, file_sr, label, *_ in dataset:
        if label != keyword:
            continue
        wav_np = waveform.squeeze().numpy()
        with tempfile.NamedTemporaryFile(suffix='.wav', delete=False) as tmp:
            tmp_path = tmp.name
        try:
            write_wav(tmp_path, wav_np, file_sr)
            feat = extract_features(tmp_path, sr=16000, n_mels=40)
        finally:
            os.unlink(tmp_path)
        feats.append(feat.numpy())
        if len(feats) >= n:
            break
    return np.stack(feats)  # (n, 40, T)

try:
    yes_feats = collect_features(train_ds, 'yes', n=10)  # (10, 40, T)
    no_feats  = collect_features(train_ds, 'no',  n=10)
except (NotImplementedError, TypeError):
    yes_feats = no_feats = None
    print('⚠️ extract_features isn\'t implemented yet — finish the TODO in section 4, then run this cell again to see the comparison plot')

if yes_feats is not None:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, feats, kw in zip(axes, [yes_feats, no_feats], ['yes', 'no']):
        im = ax.imshow(feats.mean(axis=0), origin='lower', aspect='auto', cmap='magma')
        ax.set_title(f'average mel features: "{kw}" (n=10)')
        ax.set_xlabel('time frame')
        ax.set_ylabel('mel bin')
        plt.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.show()

## 6. A custom `KWSDataset`: wrapping it all in `__getitem__`

Wrap `extract_features` + `normalize` inside a `torch.utils.data.Dataset`, so it can be handed straight to a `DataLoader`.

In [ ]:
from torch.utils.data import Dataset, DataLoader

class KWSDataset(Dataset):
    """Keyword-spotting dataset: wraps extract_features + normalize, outputs (mel_tensor, label_idx)."""

    def __init__(self, samples: list, label_map: dict, n_mels: int = 40):
        # samples: list of (waveform_np, sr, label_str)
        self.samples = samples
        self.label_map = label_map
        self.n_mels = n_mels

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        from aurora.audio.mel import mel_spectrogram, power_to_db
        x, sr, label = self.samples[idx]
        x = pad_or_truncate(x, target=sr)
        # pass n_fft/hop_length explicitly to match the documented T=32
        mel = mel_spectrogram(x, sr, n_mels=self.n_mels, n_fft=2048, hop_length=512)
        mel = mel.T  # (T, n_mels) → (n_mels, T), the (mel bins, time) layout the CNN expects
        feat = torch.tensor(normalize(power_to_db(mel)), dtype=torch.float32)
        # reject unknown labels strictly
        # background: if label isn't in label_map, label_idx would point at a position that doesn't exist (OOB = out-of-bounds).
        # e.g. label_map = {'yes': 0, 'no': 1} (2 classes), but some sample has label = 'unknown' (a 3rd class)
        # → label_idx gets mapped to some invalid value, and the CrossEntropyLoss target later goes out of range.
        # In that situation CrossEntropyLoss may not raise a clear error (a "silent error"); it can show up as unstable gradients or NaN,
        # and bugs like that are hard to track down. So we intercept it early in __getitem__ and raise an explicit exception.
        if label not in self.label_map:
            raise ValueError(f"unknown label '{label}', valid labels: {list(self.label_map)}")
        label_idx = self.label_map[label]
        return feat, label_idx

# Demo: check __getitem__ + DataLoader batch shapes with synthetic samples
import numpy as np, torch

# synthesize 16 fake samples (no real dataset needed)
LABELS = ['yes', 'no', 'up', 'down']
label_map = {l: i for i, l in enumerate(LABELS)}
fake_samples = [(np.random.randn(16000).astype(np.float32), 16000, LABELS[i % 4]) for i in range(16)]

kws_ds = KWSDataset(fake_samples, label_map, n_mels=40)
loader = DataLoader(kws_ds, batch_size=8, shuffle=True)
batch_x, batch_y = next(iter(loader))
print(f'batch shapes x: {tuple(batch_x.shape)}  y: {tuple(batch_y.shape)}')
assert batch_x.ndim == 3, 'expected (batch, n_mels, T)'
assert batch_x.shape[1] == 40, f"dim 1 should be n_mels=40, got {batch_x.shape[1]}"
assert batch_x.shape[2] > 0, "the number of time frames T should be greater than 0"
print(f'✅ KWSDataset.__getitem__ + DataLoader work correctly  batch shape: {tuple(batch_x.shape)}')

## Wrap-up

This lesson implemented `extract_features(wav_path)`, squeezing a raw WAV into a standard `(n_mels, T)` mel map; everything from `mel_spectrogram` to `power_to_db` goes through our own `aurora.audio.mel`, with no help from any torchaudio transform.
On the data side we prefer `torchaudio.datasets.SPEECHCOMMANDS` (real data, ~2.3 GB) and fall back automatically to a fixed-seed synthetic keyword dataset when there's no network or no torchaudio; both paths share the same preprocessing and the same `KWSDataset`, and the batches the `DataLoader` produces have exactly the same shape.
The next lesson (L63) builds the `KeywordCNN` convolutional classifier: it takes `(B, 1, n_mels, T)` mel feature maps and uses "kernel height = n_mels" to compress the frequency dimension in one go, defining `__init__` and `forward` from scratch.

## ✏️ Whiteboard challenge: work through the audio preprocessing pipeline by hand (target: 10 minutes)

Cover the screen and derive on paper:

**Question 1**: Which two cases does `pad_or_truncate(x, target=16000)` have to handle? What does it do in each?
(Case 1: len(x)<16000 → pad zeros at the end (np.pad or concatenate); case 2: len(x)>16000 → truncate with x[:16000]; equal length is left alone)

**Question 2**: Does `normalize(X)` use a global mean/variance or a per-frame one? Why is that better for audio features?
(Global, over the whole (n_mels,T) matrix; per-frame would destroy the energy contrast between frames, while global keeps the dynamics over time)

**Question 3**: What shape does `mel_spectrogram` output? What does it become after `.T` (transpose)?
(It outputs (n_frames, n_mels); after .T it becomes (n_mels, n_frames); a CNN expects (channels, H, W), with H=n_mels)

**Question 4**: What should `KWSDataset.__getitem__` return? What does the `DataLoader` do with it?
(It returns a (feature_tensor, label_int) pair; the DataLoader stacks B results of __getitem__ into a (B,...) batch)

**Question 5**: Why `.float()` (rather than `.double()`)? Which precision do the model parameters use?
(nn.Module defaults to float32; double() is float64, 2–8× slower on a GPU, and the extra precision is wasted on deep learning)

When you've finished the derivations, run the cell below to check.

In [ ]:
# ✏️ Answer-check cell
import numpy as np, torch

# Q1: the two cases of pad_or_truncate
x_short = np.ones(8000)
x_long  = np.ones(24000)
x_exact = np.ones(16000)
res_short = pad_or_truncate(x_short, 16000)
res_long  = pad_or_truncate(x_long,  16000)
res_exact = pad_or_truncate(x_exact, 16000)
assert len(res_short) == 16000, f"length after padding the short clip={len(res_short)}"
assert res_short[-1] == 0.0,    "the end should be zero-padded"
assert len(res_long)  == 16000, f"length after truncating the long clip={len(res_long)}"
assert len(res_exact) == 16000, f"equal length should be unchanged={len(res_exact)}"
print(f"Q1 ✅  pad_or_truncate: short→padded at the end ({len(x_short)}→{len(res_short)}), long→truncated ({len(x_long)}→{len(res_long)})")

# Q2: normalize uses a global mean/variance
X_q = np.random.default_rng(0).standard_normal((40, 32)) * 5 + 3
X_norm = normalize(X_q)
assert abs(X_norm.mean()) < 1e-6, f"mean={X_norm.mean():.6f}"
assert abs(X_norm.std() - 1.0) < 1e-5, f"std={X_norm.std():.6f}"
print(f"Q2 ✅  normalize: mean={X_norm.mean():.2e}, std={X_norm.std():.4f} (global normalization)")

# Q3: extract_features shape
try:
    import tempfile, os
    audio_test = np.random.default_rng(42).standard_normal(16000).astype(np.float32) * 0.1
    with tempfile.NamedTemporaryFile(suffix='.wav', delete=False) as f:
        tmp_q = f.name
    write_wav(tmp_q, audio_test, 16000)
    try:
        feat = extract_features(tmp_q, sr=16000, n_mels=40)
    finally:
        os.unlink(tmp_q)
    assert feat.shape[0] == 40, f"n_mels dimension={feat.shape[0]}"
    assert feat.dtype == torch.float32
    print(f"Q3 ✅  extract_features shape={tuple(feat.shape)}, dtype={feat.dtype}")
except (NotImplementedError, TypeError):
    print(f"Q3 ⬜  output is (40,T): .T turns (n_frames,40) into (40,n_frames) for the CNN (checked once extract_features is implemented)")

# Q4: DataLoader batches (proof of concept)
from torch.utils.data import TensorDataset, DataLoader
ds_q = TensorDataset(torch.randn(10, 40, 32), torch.arange(10))
dl_q = DataLoader(ds_q, batch_size=3)
batch_x, batch_y = next(iter(dl_q))
assert batch_x.shape == (3, 40, 32)
print(f"Q4 ✅  the DataLoader stacks __getitem__ results: single (40,32) → batch {tuple(batch_x.shape)}")

# Q5: float32 vs float64
t_f32 = torch.zeros(1, dtype=torch.float32)
t_f64 = torch.zeros(1, dtype=torch.float64)
assert t_f32.element_size() == 4
print(f"Q5 ✅  float32={t_f32.element_size()} bytes, matching nn.Module's default precision")
print("\n🎉 Audio preprocessing pipeline whiteboard challenge passed!")

In [ ]:
# ✏️ Self-review for this lesson
l62_review = {
    "pad_truncate_understood":  None,  # understand the two cases of making lengths uniform with pad/truncate? True/False
    "mel_shape_understood":     None,  # remember mel_spectrogram outputs (n_frames,n_mels), and (n_mels,T) after .T? True/False
    "extract_features_impl":    None,  # extract_features() implemented correctly, shape/dtype checks pass? True/False
    "dataset_dataloader":       None,  # understand __getitem__ returns one sample and DataLoader stacks them into a batch? True/False
    "whiteboard_passed":        None,  # all 5 whiteboard questions done? True/False
}

unfilled = [k for k, v in l62_review.items() if v is None]
assert not unfilled, f'not filled in yet: {unfilled}'
weak = [k for k, v in l62_review.items() if v is False]
if weak:
    print(f'⚠️  needs more work: {weak}')
else:
    print('✅ L62 fully cleared! On to L63: an audio classifier')

---

→ **Next lesson** · [L63 · An audio classifier](L63_kws_model.ipynb)

> Next lesson covers **an audio classifier**: a CNN on mel features, defining the network for Speech Commands.